# BIO-NN Experiment 1: Train Basic SNN on MNIST

Train a spiking neural network with LIF neurons on MNIST handwritten digits.
Expected: ~95% accuracy in 20 epochs on T4 GPU (~5 min).

### 1.1 Load MNIST Dataset

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_data):,}")
print(f"Test samples:  {len(test_data):,}")
print(f"Image shape:   {train_data[0][0].shape}")

### 1.2 Visualize Sample Data

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_data[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"Label: {label}", fontsize=10)
    ax.axis('off')
plt.suptitle("MNIST Sample Digits", fontsize=14)
plt.tight_layout()
plt.show()

### 1.3 Build SNN Model

In [ ]:
from bio_nn.core.model_builder import build_model

config = {
    "model": {
        "type": "bio_nn",
        "time_steps": 15,
        "input_size": 784,
        "n_classes": 10,
        "layers": [
            {"type": "lif_neuron", "size": 256},
            {"type": "lif_neuron", "size": 128}
        ]
    },
    "neuron": {
        "model": "lif",
        "tau_mem": 20.0,
        "threshold": 1.0
    },
    "synapse": {"type": "static"},
    "encoder": {"type": "rate"},
    "decoder": {"type": "rate"},
}

model = build_model(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built: {n_params:,} parameters")
print(f"Model on: {next(model.parameters()).device}")

### 1.4 Train the Model

In [ ]:
from bio_nn.training.engine import TrainingEngine

engine = TrainingEngine(model, {
    "epochs": 20,
    "lr": 1e-3,
    "timestep": 15,
    "log_interval": 5,
    "patience": 10,
    "grad_clip": 5.0
}, device)

results = engine.train(train_loader, test_loader=test_loader)

### 1.5 Plot Training Results

In [ ]:
epochs = [h['epoch'] for h in results['history']]
val_accs = [h['val_accuracy'] for h in results['history']]
train_losses = [h['train_loss'] for h in results['history']]
spikes = [h['spikes'] for h in results['history']]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, val_accs, 'b-o', markersize=4)
axes[0].set_title('Validation Accuracy', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

axes[1].plot(epochs, train_losses, 'r-o', markersize=4)
axes[1].set_title('Training Loss', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, spikes, 'g-o', markersize=4)
axes[2].set_title('Total Spikes', fontsize=12)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Spike Count')
axes[2].grid(True, alpha=0.3)

plt.suptitle('MNIST SNN Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('mnist_training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Final Test Accuracy: {results['test_results']['accuracy']:.4f}")

### 1.6 Download Results

In [ ]:
from google.colab import files
files.download('mnist_training_results.png')